In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pickle


In [ ]:
# 
# ML_analysis.ipynb
# Matt Russell
# 
# This program takes the preprocessed wide-format aggregated csv data files for eeg, fnirs, and eyetracking, and 
#    1) runs LOO-cv classification for workload / movement for all participants using random forest. 
#    2) currently uses scaling and PCA 
#

def run_ML(df):
    results = []
    
    for pid in df.pid.unique():
        scaler = StandardScaler()
        pca = PCA(n_components=0.95)

        df_train = df[df.pid != pid].reset_index(drop=True)
        df_test  = df[df.pid == pid].reset_index(drop=True)

        X_train = df_train.drop(columns=['pid', 'movement', 'block_id', 'nback'])
        y_train_workload = df_train.nback
        y_train_movement = df_train.movement

        X_test = df_test.drop(columns=['pid', 'movement', 'block_id', 'nback'])
        y_test_workload = df_test.nback
        y_test_movement = df_test.movement

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        X_train = pca.fit_transform(X_train)
        X_test = pca.transform(X_test)

        # workload classification
        clf_workload = RandomForestClassifier(n_estimators=100, random_state=42)
        clf_workload.fit(X_train, y_train_workload)
        workload_score = f1_score(y_test_workload, clf_workload.predict(X_test), average='macro', zero_division=0)

        # movement classification  
        clf_movement = RandomForestClassifier(n_estimators=100, random_state=42)
        clf_movement.fit(X_train, y_train_movement)
        movement_score = f1_score(y_test_movement, clf_movement.predict(X_test), average='macro', zero_division=0)

        # Append results for this participant
        results.append({'pid': pid, 'task': 'workload', 'f1_score': workload_score})
        results.append({'pid': pid, 'task': 'movement', 'f1_score': movement_score})
    
    results_df = pd.DataFrame(results)
    results_df = results_df.set_index(['task', 'pid'])['f1_score']
    
    return results_df, clf_workload, scaler

all_results = []
#THESE ARE THE STREAM NAMES
#TODO: can we add EEG
for dname in ['OxySoft']: 
    df = pd.read_csv(f'../data/processed_data/processed_agg_{dname}_wide.csv') 

    ml_results, model, scaler = run_ML(df)
    ml_results.name = dname
    all_results.append(ml_results)

    # After training
    with open("model.pkl", "wb") as f:
        pickle.dump(model, f)

    with open("scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)


final_results = pd.concat(all_results, axis=1)
final_results.groupby('task').describe().T